# Hourly demand estimation

Rental operations need hourly demand estimates for staffing and availability. Random train/test splits can hide time drift and make a model look better than it will on a future period.

Train on 2011, select and estimate residual widths on Jan–Jun 2012, and test on Jul–Dec 2012. Compare Poisson and boosted regressors to a historical hour/weekday baseline, add rolling training checks and report error by hour.

Rebuild source artifacts with `python scripts/run_all.py` from the repository root. This walkthrough inspects the measured run.

In [1]:
from pathlib import Path
import json
root = Path.cwd()
if root.name == "notebooks":
    root = root.parent
filename = "metrics.json"
report = json.loads((root / "artifacts/demand" / filename).read_text(encoding="utf-8"))
print(json.dumps(report.get("test", report.get("decisions", report.get("aggregate"))), indent=2))

{
  "mae": 94.07015031656223,
  "rmse": 130.81954455777307,
  "wape": 0.37022268869735375,
  "empirical_interval_coverage": 0.8800274223034735,
  "mean_interval_width": 340.99891955650395
}


In [2]:
import pandas as pd
result = pd.read_csv(root / "artifacts/demand/holdout_predictions.csv")
print(result.head(8).to_string(index=False))

          timestamp  cnt  weathersit  hr  predicted  lower      upper  absolute_error
2012-07-01 00:00:00  149           1   0 104.929858    0.0 315.570797       44.070142
2012-07-01 01:00:00   93           1   1  84.352098    0.0 294.993037        8.647902
2012-07-01 02:00:00   90           1   2  63.081659    0.0 273.722597       26.918341
2012-07-01 03:00:00   33           1   3  30.687003    0.0 241.327942        2.312997
2012-07-01 04:00:00    4           1   4  21.167435    0.0 231.808373       17.167435
2012-07-01 05:00:00   10           1   5  17.037109    0.0 227.678048        7.037109
2012-07-01 06:00:00   27           1   6  13.613149    0.0 224.254087       13.386851
2012-07-01 07:00:00   50           1   7  52.873123    0.0 263.514062        2.873123


## Interpretation and limits

Uses observed weather: conditional estimation rather than a weather forecast. Temporal drift means the nominal 90% intervals have no coverage guarantee. The measured coverage is reported. No claim of performance in a new city or fleet.